# Proyecto: Optimización de Hiperparámetros con Metaheurísticas

## Objetivo general

El objetivo de este proyecto es implementar y comparar métodos de optimización metaheurística para ajustar hiperparámetros de modelos de aprendizaje automático.

Los estudiantes deberán estudiar y aplicar:

- recocido simulado,
- algoritmos evolutivos,
- búsqueda estocástica de arquitecturas,
- optimización de hiperparámetros.

El proyecto busca integrar:

- optimización,
- machine learning,
- redes neuronales,
- búsqueda estocástica,
- validación,
- y selección de modelos.

---

# 1. Motivación

En muchos modelos de machine learning, el rendimiento depende fuertemente de hiperparámetros como:

- tasa de aprendizaje,
- regularización,
- número de capas,
- número de neuronas,
- funciones de activación,
- optimizador.

Encontrar buenas combinaciones suele ser costoso.

---

# 2. Problema de optimización

Sea:

$$
\theta
$$

el vector de hiperparámetros.

El objetivo es resolver:

$$
\theta^*
=
\arg\min_{\theta}
L(\theta)
$$

donde:

- $L(\theta)$ representa la pérdida de validación.

---

# 3. Espacio de búsqueda

Los hiperparámetros pueden incluir:

---

## Arquitectura

- número de capas,
- número de neuronas por capa.

---

## Optimización

- learning rate,
- weight decay,
- optimizador.

---

## Activación

Ejemplos:

- ReLU,
- Tanh,
- GELU,
- ELU.

---

# 4. Recocido simulado

El recocido simulado es una metaheurística inspirada en procesos termodinámicos.

Permite escapar de óptimos locales mediante movimientos aleatorios controlados por temperatura.

---

# 5. Estado

Cada estado representa una configuración:

$$
s
=
\theta
$$

es decir, una combinación de hiperparámetros.

---

# 6. Función objetivo

La energía del sistema es:

$$
E(s)=L(s)
$$

donde:

- menor energía implica mejor configuración.

---

# 7. Movimiento

A partir de un estado actual se propone uno nuevo mediante perturbaciones:

- modificar learning rate,
- cambiar optimizador,
- modificar capas,
- cambiar neuronas,
- cambiar activación.

---

# 8. Regla de aceptación

Si el nuevo estado mejora:

$$
E(s')<E(s)
$$

se acepta.

En otro caso se acepta con probabilidad:

$$
P=
\exp\left(
-\frac{E(s')-E(s)}{T}
\right)
$$

---

# 9. Enfriamiento

La temperatura evoluciona como:

$$
T_{t+1}
=
\alpha T_t
$$

con:

$$
0<\alpha<1
$$

Esto reduce exploración gradualmente.

---

# 10. Algoritmos evolutivos

Los algoritmos evolutivos simulan evolución biológica.

Cada solución representa un individuo.

---

# 11. Población

La población inicial consiste en múltiples configuraciones:

$$
\{s_1,\dots,s_m\}
$$

cada una con hiperparámetros distintos.

---

# 12. Fitness

La aptitud se define mediante:

$$
f(s)
=
-\text{Loss}(s)
$$

mayor fitness implica mejor solución.

---

# 13. Selección

Se seleccionan padres según su desempeño.

En este proyecto se utiliza selección probabilística tipo softmax.

---

# 14. Cruce

Se combinan dos padres para generar descendencia.

El hijo hereda:

- arquitectura,
- optimizador,
- learning rate,
- activación.

---

# 15. Mutación

Se aplican perturbaciones aleatorias para explorar nuevas regiones.

Ejemplos:

- modificar número de capas,
- alterar learning rate,
- cambiar activación.

---

# 16. Elitismo

Los mejores individuos se preservan entre generaciones.

Esto evita perder soluciones de alta calidad.

---

# 17. Red neuronal

El modelo base es una red multicapa:

$$
h^{(1)}=\phi(XW_1+b_1)
$$

$$
h^{(2)}=\phi(h^{(1)}W_2+b_2)
$$

$$
\hat y = f(x)
$$

---

# 18. Entrenamiento

Cada configuración requiere:

1. construir red,
2. entrenar modelo,
3. validar desempeño,
4. calcular pérdida.

Esto hace el problema computacionalmente costoso.

---

# 19. Comparación entre métodos

Los estudiantes deberán comparar:

---

## Recocido Simulado

Ventajas:

- sencillo,
- buena exploración,
- fácil implementación.

---

## Algoritmo Evolutivo

Ventajas:

- búsqueda paralela,
- diversidad,
- buena exploración global.

---

# 20. Métricas

Se recomienda evaluar:

---

## Accuracy

$$
\text{Accuracy}
=
\frac{1}{n}
\sum_{i=1}^{n}
I(y_i=\hat y_i)
$$

---

## Pérdida de validación

- cross-entropy,
- NLL.

---

## Tiempo computacional

Comparar costo de ambos métodos.

---

# 21. Extensiones opcionales

Los estudiantes pueden agregar:

- paralelización,
- búsqueda híbrida,
- GPU,
- early stopping,
- ensembles.

---

# 22. Objetivos específicos

Los estudiantes deberán:

1. Seleccionar un dataset.
2. Implementar red base.
3. Definir espacio de hiperparámetros.
4. Implementar recocido simulado.
5. Implementar algoritmo evolutivo.
6. Comparar resultados.
7. Analizar convergencia.
8. Interpretar desempeño.
9. Elaborar conclusiones.

---

# 23. Entregables sugeridos

El proyecto deberá incluir:

- notebook reproducible,
- implementación de ambos métodos,
- análisis teórico,
- comparación experimental,
- visualización de convergencia,
- análisis de resultados,
- conclusiones.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

import numpy as np
import random
from copy import deepcopy
from tqdm.auto import tqdm

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# **Reconocido Simulado**

In [2]:
activations = {"relu": nn.ReLU(), "tanh": nn.Tanh(), "gelu": nn.GELU(), "elu":nn.ELU()}

In [3]:
class MLP(nn.Module):

    def __init__(self, input_dim, output_dim, hidden_dims, activation) :

        super().__init__()

        inp_prev = input_dim

        activation = activations[activation]

        layers = []

        for h in hidden_dims:

            layers.append(nn.Linear(inp_prev, h))
            layers.append(activation)
            inp_prev = h

        layers.append(nn.Linear(inp_prev, output_dim))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [4]:
def train_model(model,
                train_loader,
                valid_loader,
                optimizer_name,
                lr,
                weight_decay,
                epochs=50,
                device="cpu"):

    model.to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    else:

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay
        )

    for _ in range(epochs):

        model.train()

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)

            loss = criterion(logits, y)

            loss.backward()

            optimizer.step()

    model.eval()

    total_loss = 0
    n = 0

    with torch.no_grad():

        for x, y in valid_loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * len(x)

            n += len(x)

    return total_loss / n

In [5]:
def random_state():

    n_layers = random.randint(1, 5)

    hidden_dims = [
        random.choice([4, 8, 16, 32, 64, 128, 256])
        for _ in range(n_layers)
    ]

    return {
        "lr": 10**np.random.uniform(-4, -1),
        "weight_decay": 10**np.random.uniform(-6, -2),
        "optimizer": random.choice(["adam", "sgd"]),
        "activation": random.choice(
            ["relu", "tanh", "gelu", "elu"]
        ),
        "hidden_dims": hidden_dims
    }

In [6]:
def transition_kernel(state):

    new_state = deepcopy(state)

    move = random.choice([
        "lr",
        "weight_decay",
        "optimizer",
        "activation",
        "layers",
        "neurons"
    ])

    # LR
    if move == "lr":

        log_lr = np.log(state["lr"]/(1-state["lr"]))

        log_lr += np.random.normal(0, 0.5)

        log_lr = np.clip(log_lr, -6, -1)

        new_state["lr"] = 1/(1+ np.exp(log_lr))

    # L2
    elif move == "weight_decay":

        log_wd = np.log(state["weight_decay"])

        log_wd += np.random.normal(0, 0.5)

        log_wd = np.clip(log_wd, -7, 1)

        new_state["weight_decay"] = np.exp(log_wd)

    # optimizer
    elif move == "optimizer":

        new_state["optimizer"] = (
            "adam"
            if state["optimizer"] == "sgd"
            else "sgd"
        )

    # activation
    elif move == "activation":

        acts = ["relu", "tanh", "gelu", "elu"]

        acts.remove(state["activation"])

        new_state["activation"] = random.choice(acts)

    # numero de capas
    elif move == "layers":

        current = len(state["hidden_dims"])

        delta = random.choice([-1, 1])

        new_layers = np.clip(current + delta, 1, 5)

        hidden = deepcopy(state["hidden_dims"])

        if new_layers > current:

            hidden.append(
                random.choice([4, 8, 16, 32, 64, 128, 256])
            )

        else:

            hidden = hidden[:new_layers]

        new_state["hidden_dims"] = hidden

    # neuronas
    elif move == "neurons":

        hidden = deepcopy(state["hidden_dims"])

        # elegir capa aleatoria
        idx = random.randint(
            0,
            len(hidden)-1
        )

        # movimiento multiplicativo
        factor = random.choice([
            0.5,
            2
        ])

        hidden[idx] = int(
            hidden[idx] * factor
        )

        # restringir tamaño
        hidden[idx] = int(
            np.clip(hidden[idx], 4, 512)
        )

        new_state["hidden_dims"] = hidden

    return new_state

In [7]:
def simulated_annealing(
        train_loader,
        valid_loader,
        input_dim,
        output_dim,
        iterations=50,
        T0=1.0,
        alpha=0.95,
        device="cpu"):

    # =========================
    # Estado inicial
    # =========================

    current = random_state()

    model = MLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dims=current["hidden_dims"],
        activation=current["activation"]
    )

    current_loss = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        optimizer_name=current["optimizer"],
        lr=current["lr"],
        weight_decay=current["weight_decay"],
        device=device
    )

    # =========================
    # Mejor solución encontrada
    # =========================

    best_state = deepcopy(current)

    best_loss = current_loss

    # GUARDAR PESOS
    best_model_state = deepcopy(
        model.state_dict()
    )

    T = T0

    # tqdm
    pbar = tqdm(
        range(iterations),
        desc="Simulated Annealing"
    )

    for k in pbar:

        # =========================
        # Proposal
        # =========================

        proposal = transition_kernel(current)

        model = MLP(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=proposal["hidden_dims"],
            activation=proposal["activation"]
        )

        proposal_loss = train_model(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            optimizer_name=proposal["optimizer"],
            lr=proposal["lr"],
            weight_decay=proposal["weight_decay"],
            device=device
        )

        # =========================
        # Acceptance step
        # =========================

        delta = proposal_loss - current_loss

        accept = False

        if delta < 0:

            accept = True

        else:

            prob = np.exp(-delta / T)

            if np.random.rand() < prob:

                accept = True

        # =========================
        # Update chain
        # =========================

        if accept:

            current = deepcopy(proposal)

            current_loss = proposal_loss

            # =========================
            # Mejor modelo global
            # =========================

            if current_loss < best_loss:

                best_loss = current_loss

                best_state = deepcopy(current)

                # GUARDAR PESOS
                best_model_state = deepcopy(
                    model.state_dict()
                )

        # =========================
        # Cooling
        # =========================

        T *= alpha

        # =========================
        # tqdm info
        # =========================

        pbar.set_postfix({

            "loss": f"{current_loss:.4f}",

            "best": f"{best_loss:.4f}",

            "T": f"{T:.4f}",

            "lr": f"{current['lr']:.2e}",

            "wd": f"{current['weight_decay']:.2e}",

            "opt": current["optimizer"],

            "act": current["activation"],

            "layers": len(current["hidden_dims"]),

            "hidden": str(current["hidden_dims"]),

            "accept": int(accept)

        })

    # =========================
    # Reconstruir mejor modelo
    # =========================

    best_model = MLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dims=best_state["hidden_dims"],
        activation=best_state["activation"]
    )

    best_model.load_state_dict(
        best_model_state
    )

    # =========================
    # Return
    # =========================

    return best_model, best_state, best_loss

In [8]:
wine = load_wine()

X = wine.data
y = wine.target

scaler = StandardScaler()

X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.long
)


train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


In [9]:
input_dim = X_train.shape[1]
output_dim = len(torch.unique(y_train))


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
torch.manual_seed(18081997)
best_model, best_hparams, best_loss = simulated_annealing(
    train_loader=train_loader,
    valid_loader=test_loader,
    input_dim=input_dim,
    output_dim=output_dim,
    iterations=150,
    T0=1.0,
    alpha=0.95,
    device=device
)

Simulated Annealing:   0%|          | 0/150 [00:00<?, ?it/s]

In [12]:
best_loss

0.039665851308503385

In [13]:
best_model

MLP(
  (net): Sequential(
    (0): Linear(in_features=13, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=32, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=32, out_features=64, bias=True)
    (5): GELU(approximate='none')
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): GELU(approximate='none')
    (8): Linear(in_features=32, out_features=3, bias=True)
  )
)

In [14]:
best_hparams

{'lr': 0.005845185729920755,
 'weight_decay': np.float64(0.0009118819655545162),
 'optimizer': 'adam',
 'activation': 'gelu',
 'hidden_dims': [256, 32, 64, 32]}

In [15]:
y_pred = torch.softmax(best_model.forward(X_test), dim=-1).argmax(dim=-1)
(y_pred == y_test).float().mean()

tensor(0.9722)

# **Algoritmo Evolutivo**

In [11]:
def evaluate_fitness(state,
                      train_loader,
                      valid_loader,
                      input_dim,
                      output_dim,
                      device="cpu"):

    model = MLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dims=state["hidden_dims"],
        activation=state["activation"]
    )

    loss = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        optimizer_name=state["optimizer"],
        lr=state["lr"],
        weight_decay=state["weight_decay"],
        epochs=10,   # importante: GA usa pocas epochs
        device=device
    )

    # GA maximiza fitness
    return -loss

In [12]:
def softmax_selection(population, fitnesses, n_parents):

    fitnesses = np.array(fitnesses, dtype=float)

    probs = np.exp(fitnesses - np.max(fitnesses))
    probs = probs / probs.sum()

    idx = np.random.choice(
        len(population),
        size=n_parents,
        replace=True,
        p=probs
    )

    return [deepcopy(population[i]) for i in idx]

In [13]:
def crossover(parent1, parent2):

    child = {}

    for key in parent1.keys():

        if key == "hidden_dims":
            # crossover especial para arquitectura
            if random.random() < 0.5:
                child[key] = deepcopy(parent1[key])
            else:
                child[key] = deepcopy(parent2[key])
        else:
            child[key] = (
                deepcopy(parent1[key])
                if random.random() < 0.5
                else deepcopy(parent2[key])
            )

    return child

In [14]:
def mutate(state, mutation_prob=0.3):

    if random.random() < mutation_prob:
        return transition_kernel(state)

    return state

In [15]:
def init_population(pop_size=20):
    return [random_state() for _ in range(pop_size)]

In [16]:
def genetic_algorithm(
        train_loader,
        valid_loader,
        input_dim,
        output_dim,
        generations=30,
        pop_size=10,
        elite_size=2,
        mutation_prob=0.3,
        device="cpu"
    ):

    population = init_population(pop_size)

    best_state = None
    best_fitness = -np.inf
    history = []

    pbar = tqdm(range(generations), desc="Genetic Algorithm")

    for gen in pbar:

        # =========================
        # 1. EVALUACIÓN (SECUENCIAL)
        # =========================
        fitnesses = [
            evaluate_fitness(
                ind,
                train_loader,
                valid_loader,
                input_dim,
                output_dim,
                device
            )
            for ind in population
        ]

        # =========================
        # 2. MEJOR GLOBAL
        # =========================
        gen_best_idx = np.argmax(fitnesses)

        if fitnesses[gen_best_idx] > best_fitness:
            best_fitness = fitnesses[gen_best_idx]
            best_state = deepcopy(population[gen_best_idx])

        history.append(best_fitness)

        # =========================
        # 3. ELITISMO
        # =========================
        elite_idx = np.argsort(fitnesses)[-elite_size:]
        no_elite_indx = np.argsort(fitnesses)[:elite_size]

        no_elite_population = [
            deepcopy(population[i])
            for i in no_elite_indx
        ]

        elite_population = [
            deepcopy(population[i])
            for i in elite_idx
        ]

        new_population = []


        # =========================
        # 4. GENERACIÓN NUEVA
        # =========================
        while len(new_population) < pop_size:

            parents_1 = np.random.choice(elite_population, 1)
            parents_2 = np.random.choice(no_elite_population, 1)

            child = crossover(parents_1[0], parents_2[0])

            child = mutate(child, mutation_prob)

            new_population.append(child)

        population = new_population

        # =========================
        # tqdm info
        # =========================
        pbar.set_postfix({
            "best_fitness": best_fitness,
            "layers": len(best_state["hidden_dims"]),
            "lr": f'{best_state["lr"]:.2e}',
            "opt": best_state["optimizer"],
            "act": best_state["activation"]
        })

    return {
        "best_state": best_state,
        "best_fitness": best_fitness,
        "history": history
    }

In [21]:
input_dim = X_train.shape[1]
output_dim = len(torch.unique(y_train))

torch.random.manual_seed(11256)
results = genetic_algorithm(
    train_loader,
    test_loader,
    input_dim=input_dim,
    output_dim=output_dim,
    generations=50,
    pop_size=10,
    elite_size=2,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

Genetic Algorithm:   0%|          | 0/50 [00:00<?, ?it/s]

In [22]:
results["best_state"]

{'lr': np.float64(0.9559570419570079),
 'weight_decay': np.float64(0.0009118819655545162),
 'optimizer': 'sgd',
 'activation': 'elu',
 'hidden_dims': [64]}

In [23]:
results["best_fitness"]

-0.0

In [24]:
results["history"]

[-0.058854791643372865,
 -0.05480544869432277,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -1.867132634616711e-05,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0,
 -0.0]